# Jupyter Notebook
**Projekt:** The Hidden Cost of Weather  
**Fáze:** Prepare (Extrakce surových dat)  
**Autor:** Lukáš Weissgráb

Tento Jupyter Notebook obsahuje podrobnou cestu datového projektu, který se týká vlivu počasí na zpoždění letů v USA.

## 01 - Data Extraction (METAR zprávy & Delays)


Importujeme důležité knihovny pro naší analýzu.

In [ ]:
import os
from pathlib import Path
import time
import urllib.parse
import zipfile
import pandas as pd
import numpy as np
import requests

# Příprava složek v souladu s architekturou projektu
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Úložiště připraveno: {RAW_DIR.resolve()}")

Úložiště připraveno: /content/data/raw


Nyní nás čeká import jednotlivých dat. Začneme importem METAR zpráv, což jsou zprávy týkající se počasí na daném letišti (zahrnují čas vydání, vítr, teplotu, rosný bod, dohlednost a význačné jevy počasí - déšť, bouřka apod.).

In [ ]:
# -------------------------------------------------------------
# STAŽENÍ A SPOJENÍ METAR ZPRÁV (IEM ASOS / ČERVENEC 2023)
# -------------------------------------------------------------
airports = ["ORD", "ATL", "DFW", "JFK"]
dataframes = []

for airport in airports:
    # Parametry: stanice, rozsah 1. 7. až 31. 7. 2023, čas UTC
    url = (
        f"https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?"
        f"station={airport}&data=metar&year1=2023&month1=7&day1=1"
        f"&year2=2023&month2=7&day2=31&tz=Etc/UTC&format=onlycomma"
    )

    csv_path = RAW_DIR / f"metar_{airport.lower()}_2023_07.csv"

    # Stažení a uložení do složky data/raw
    response = requests.get(url)
    with open(csv_path, "w", encoding="utf-8") as f:
        f.write(response.text)

    # Načtení do paměti a připojení k celku
    df_temp = pd.read_csv(csv_path)
    dataframes.append(df_temp)
    print(f"K{airport}: staženo {len(df_temp):,} zpráv")

# Sloučení všech 4 stanic do jedné tabulky
df_metar = pd.concat(dataframes, ignore_index=True)

# Kontrola výsledku
print(f"\nCelkem METAR zpráv v tabulce: {len(df_metar):,}")
df_metar.head()

KORD: staženo 9,432 zpráv
KATL: staženo 9,362 zpráv
KDFW: staženo 9,321 zpráv
KJFK: staženo 8,665 zpráv

Celkem METAR zpráv v tabulce: 36,780


,station,valid,metar
0,ORD,2023-07-01 00:00,KORD 010000Z AUTO 26005KT 9SM FEW045 SCT100 29...
1,ORD,2023-07-01 00:05,KORD 010005Z AUTO 28005KT 9SM FEW045 SCT100 29...
2,ORD,2023-07-01 00:10,KORD 010010Z AUTO 27005KT 9SM FEW045 SCT100 29...
3,ORD,2023-07-01 00:15,KORD 010015Z AUTO 26005KT 9SM FEW045 SCT100 29...
4,ORD,2023-07-01 00:20,KORD 010020Z AUTO 25005KT 9SM FEW045 SCT100 29...


V dalším kroku provedeme import leteckých dat a informací o zpoždění (a jeho typu) v červenci 2023.

In [ ]:
# -------------------------------------------------------------
# STAŽENÍ A ROZBALENÍ BTS ON-TIME PERFORMANCE (07/2023)
# -------------------------------------------------------------
bts_url = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2023_7.zip"
zip_path = RAW_DIR / "bts_2023_07.zip"
csv_path = RAW_DIR / "bts_2023_07.csv"

# 1. Stažení ZIP archivu (cca 30 MB)
print("Stahuji BTS letová data z repozitáře USDOT...")
headers = {"User-Agent": "Mozilla/5.0"}
res = requests.get(bts_url, headers=headers, stream=True, timeout=180)
res.raise_for_status()

with open(zip_path, "wb") as f:
    for chunk in res.iter_content(chunk_size=1024 * 1024):
        if chunk:
            f.write(chunk)

# 2. Rozbalení CSV do složky data/raw a smazání staženého ZIPu
with zipfile.ZipFile(zip_path, "r") as z:
    raw_name = [name for name in z.namelist() if name.endswith(".csv")][0]
    z.extract(raw_name, RAW_DIR)
    os.rename(RAW_DIR / raw_name, csv_path)

zip_path.unlink()
print(f"Hotovo! Rozbaleno do: {csv_path.name}")

# 3. Rychlá kontrola prvních řádků
df_bts_sample = pd.read_csv(csv_path, nrows=3)
print(f"Tabulka má {len(df_bts_sample.columns)} sloupců.")
df_bts_sample.iloc[:, :6]

Stahuji BTS letová data z repozitáře USDOT...
Hotovo! Rozbaleno do: bts_2023_07.csv
Tabulka má 110 sloupců.


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate
0,2023,3,7,2,7,2023-07-02
1,2023,3,7,5,3,2023-07-05
2,2023,3,7,9,7,2023-07-09


Závěrem ještě zjistíme s jak velkými datovými soubory pracujeme.

In [ ]:
# -------------------------------------------------------------
# VELIKOST A POČET ŘÁDKŮ SUROVÝCH DAT (PREPARE)
# -------------------------------------------------------------
file_stats = []

for file_path in sorted(RAW_DIR.glob("*.csv")):
    size_mb = file_path.stat().st_size / (1024 * 1024)
    # Rychlý součet řádků bez načítání celého souboru do paměti
    with open(file_path, "rb") as f:
        row_count = sum(1 for _ in f) - 1  # Odečteme hlavičku

    file_stats.append(
        {
            "Soubor": file_path.name,
            "Velikost (MB)": round(size_mb, 2),
            "Počet záznamů": f"{row_count:,}",
        }
    )

df_file_stats = pd.DataFrame(file_stats)
df_file_stats

,Soubor,Velikost (MB),Počet záznamů
0,bts_2023_07.csv,259.93,"601,866"
1,metar_atl_2023_07.csv,0.90,"9,362"
2,metar_dfw_2023_07.csv,0.84,"9,321"
3,metar_jfk_2023_07.csv,0.81,"8,665"
4,metar_ord_2023_07.csv,0.90,"9,432"


## 02 - PROCESS: Čištění dat, příprava METAR zpráv a filtrování letů


In [6]:
# Cesta k surovému souboru z BTS
RAW_DIR = Path("data/raw")
bts_file = RAW_DIR / "bts_2023_07.csv"

# 1. Výběr pouze relevantních sloupců pro analýzu zpoždění a rotací
columns_to_keep = [
    "FlightDate", "Reporting_Airline", "Tail_Number", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "DepTime", "DepDelay",
    "CRSArrTime", "ArrTime", "ArrDelay", "Cancelled",
    "CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"
]

print("Načítám vybrané sloupce z BTS dat...")
df_flights = pd.read_csv(bts_file, usecols=columns_to_keep, low_memory=False)
initial_rows = len(df_flights)
print(f"Původní počet záznamů za celé USA: {initial_rows:,}")

# 2. Odstranění zrušených letů (Cancelled == 1)
# Zrušené lety neproběhly, nemají reálné letové časy a netvoří rotaci stroje
df_flights = df_flights[df_flights["Cancelled"] == 0].copy()
cancelled_diff = initial_rows - len(df_flights)
print(f"Odstraněno zrušených letů: {cancelled_diff:,}")

# 3. Odstranění letů bez registrace letadla (Tail_Number is NaN)
# Bez imatrikulace není možné let zařadit do řetězce rotace konkrétního stroje
df_flights = df_flights.dropna(subset=["Tail_Number"]).copy()

# 4. Ošetření chybějících hodnot v kategoriích zpoždění
# BTS vykazuje NaN, pokud zpoždění nevzniklo nebo bylo pod 15 minut; nahradíme nulou
delay_cols = ["CarrierDelay", "WeatherDelay", "NASDelay", "LateAircraftDelay"]
df_flights[delay_cols] = df_flights[delay_cols].fillna(0)

print(f"Konečný počet platných záznamů po základní očistě: {len(df_flights):,}")
df_flights.head(3)

Načítám vybrané sloupce z BTS dat...
Původní počet záznamů za celé USA: 601,866
Odstraněno zrušených letů: 14,606
Konečný počet platných záznamů po základní očistě: 587,260


,FlightDate,Reporting_Airline,Tail_Number,Flight_Number_Reporting_Airline,Origin,Dest,CRSDepTime,DepTime,DepDelay,CRSArrTime,ArrTime,ArrDelay,Cancelled,CarrierDelay,WeatherDelay,NASDelay,LateAircraftDelay
0,2023-07-02,9E,N307PQ,4900,DTW,DSM,946,942.0,-4.0,1034,1015.0,-19.0,0.0,0.0,0.0,0.0,0.0
1,2023-07-05,9E,N232PQ,4900,DTW,DSM,1355,1356.0,1.0,1442,1431.0,-11.0,0.0,0.0,0.0,0.0,0.0
2,2023-07-09,9E,N480PX,4900,DTW,DSM,946,942.0,-4.0,1034,1018.0,-16.0,0.0,0.0,0.0,0.0,0.0
